# Agent orchestration

Orchestration refers to the flow of agents in your app. Which agents run, in what order, and how is the next step decided?

[Documentation](https://openai.github.io/openai-agents-python/multi_agent/)

### 1. Agents as tools

In some workflows, you may want a central agent to orchestrate a network of specialized agents, instead of handing off control. You can do this by modeling agents as tools.


In [2]:
from agents import (Agent, 
                    Runner, AsyncOpenAI, 
                    OpenAIChatCompletionsModel, set_tracing_disabled)

from agents.decorators import tool

In [3]:
# Tracing disabled
set_tracing_disabled(disabled=True)

In [4]:
import os
gemini_api = os.environ['GEMINI_API_KEY']

In [5]:
# 1. Which LLM Service?
external_client = AsyncOpenAI(
    api_key=gemini_api,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [6]:
# 2. Which LLM Model?
llm_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite",
    openai_client=external_client
)

In [8]:
# Defining Agent-1
spanish_agent = Agent(
    name="Spanish agent",
    instructions="You translate the user's message to Spanish",
    model = llm_model
)

# Defining Agent-2
french_agent = Agent(
    name="French agent",
    instructions="You translate the user's message to French",
    model = llm_model
)

In [ ]:
orchestrator_agent = Agent(
    name="orchestrator_agent",
    instructions='''You are a translation agent. You use the tools given to you to translate. 
                    If asked for multiple translations, you call the relevant tools.''',
    model = llm_model,
    tools=[spanish_agent.as_tool(
            tool_name="translate_to_spanish",
            tool_description="Translate the user's message to Spanish",
        ),
        french_agent.as_tool(
            tool_name="translate_to_french",
            tool_description="Translate the user's message to French",
        ),
    ],
)

In [10]:
result = await Runner.run(orchestrator_agent, input="Say 'Hello, how are you?' in Spanish.")

print("\nCALLING AGENT\n")
print(result.final_output)


CALLING AGENT

Hola, ¿cómo estás?


In [19]:
from agents import ToolCallItem, ToolCallOutputItem

for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print("TOOL CALLED:", item.raw_item.name)
        print("ARGUMENTS:  ", item.raw_item.arguments)
    elif isinstance(item, ToolCallOutputItem):
        print("TOOL OUTPUT:", item.output)

TOOL CALLED: translate_to_spanish
ARGUMENTS:   {"input":"Hello, how are you?"}
TOOL OUTPUT: Hola, ¿cómo estás?


### 2. Handoffs

Handoffs allow an agent to delegate tasks to another agent.

[Documentation](https://openai.github.io/openai-agents-python/handoffs/)

In [46]:
from agents import Agent

history_tutor_agent = Agent(
    name="History Tutor",
    handoff_description="Specialist agent for historical questions",
    instructions="You answer history questions clearly and concisely.",
    model = llm_model
)

math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math questions",
    instructions="You explain math step by step and include worked examples.",
    model = llm_model
)

In [40]:
triage_agent = Agent(
    name="Triage Agent",
    instructions="Route each homework question to the right specialist.",
    handoffs=[history_tutor_agent, math_tutor_agent],
    model = llm_model
)

In [41]:
result = await Runner.run(triage_agent, input="Who was the first president of the Pakistan?")

print("\nCALLING AGENT\n")
print(result.final_output)


CALLING AGENT

The first president of Pakistan was **Iskander Mirza**. He served from 1956 to 1958, following the adoption of the country's first constitution.


In [42]:
print('Agent Executed at the end:', result.last_agent.name)

Agent Executed at the end: History Tutor


In [43]:
from agents import HandoffCallItem

for item in result.new_items:
    if isinstance(item, HandoffCallItem):
        print("Tool name called:", item.raw_item.name)

Tool name called: transfer_to_history_tutor


##### Customizing handoffs via the handoff() function
[Documentation](https://openai.github.io/openai-agents-python/handoffs/#customizing-handoffs-via-the-handoff-function)

In [53]:
from agents import Agent, Runner, handoff

history_tutor_agent = Agent(
    name="History Tutor Agent",
    instructions=(
        "You are a history tutor. Explain historical events clearly and accurately, "
        "including key dates, causes, and consequences. Keep answers concise and easy to follow."
    ),
)

orchestrator_agent2 = Agent(
    name="Orchestrator",
    instructions="For any history-related question, hand off to the History Tutor agent.",
    handoffs=[
        handoff(
            agent=history_tutor_agent,
            tool_name_override="ask_history_tutor",
            tool_description_override="Use this for any question about historical events, people, or periods.",
        )
    ],
)

result = await Runner.run(
    orchestrator_agent2,
    input="What were the main causes of World War I?",
)

print(result.final_output)
print("Final agent:", result.last_agent.name)

The main causes of World War I were:

- **Militarism:** European powers built large armies and navies, creating rivalry and fear.
- **Alliance systems:** The **Triple Alliance** (Germany, Austria-Hungary, Italy) and **Triple Entente** (France, Russia, Britain) meant a conflict could spread quickly.
- **Imperialism:** Competition for colonies increased tensions, especially between Germany, France, and Britain.
- **Nationalism:** Ethnic and national groups demanded independence, particularly in the Balkans; Serbian nationalism threatened Austria-Hungary.
- **Balkan instability:** Rivalries among Balkan states and the decline of the Ottoman Empire created repeated crises.
- **Immediate trigger:** On **June 28, 1914**, Serbian nationalist **Gavrilo Princip** assassinated Archduke **Franz Ferdinand** of Austria-Hungary. Austria-Hungary declared war on Serbia on **July 28**, activating alliances and starting the war.
Final agent: History Tutor Agent


In [54]:
from agents import HandoffCallItem

for item in result.new_items:
    if isinstance(item, HandoffCallItem):
        print("Tool name called:", item.raw_item.name)

Tool name called: ask_history_tutor
